# Attaching Data Assets (raw and processed)
- ophys:
    - Stage_0 (1 session): Natural movies
    - Stage_1 (3 sessions): drifting gratings

- coregistration:
    -   ophys_zstack:
        - Ophys functional (4 sessions, 8 planes) to ophys structure (1 3D image 0-400um) mapping
    - xenium_zstack:
        - Xenium to ophys strucure mapping
- xenium:
    - processed:
        - images
        - segmentation masks
        - cell locations
        - transcript locations
        - cellxgene table
    - cell_types:
        - cell types for each cell (based on ABC atlas)
        - cell typting measures

In [1]:
import sys
sys.path.append('/root/capsule/code/Preprocessing')

import codeocean
import codeocean.data_asset

from codeocean.data_asset import (
        DataAssetParams
        )
from codeocean.data_asset import DataAssetAttachParams
import aind_session
from aind_session.utils.codeocean_utils import get_data_asset_source_dir, get_codeocean_client, search_data_assets, get_data_asset_search_query, get_subject_data_assets
from analysis_utils import get_running_timestamps, get_running_df, get_stimulus_timestamps
from codeocean.data_asset import DataAssetAttachParams
import boto3
import json
import numpy as np
from pathlib import Path
import pandas as pd

import pickle
DATA_DIR = Path("/root/capsule/data")

In [2]:
client = get_codeocean_client()

In [4]:
xenium_mouse_ids = [778174,786297,797371]

co_assets_dict = {}
for mouse_id in xenium_mouse_ids:
    co_assets_dict[mouse_id] = {}
    co_assets_dict[mouse_id]['ophys'] = {}
    assets = list(get_subject_data_assets(mouse_id))
    assets.sort(key=lambda x: x.name)
    # search_params["query"] = get_data_asset_search_query(mouse_id=mouse_id)
    # from_co = search_data_assets(search_params) + search_data_assets(
    #     {"query": str(mouse_id)}
    # )
    assets_raw = [asset for asset in assets if 'multiplane-ophys' in asset.tags and 'raw' in asset.tags and 'multisession' not in asset.tags]

    for asset_raw in assets_raw:
        s3_dir = get_data_asset_source_dir(asset_raw.id).as_posix()
        s3 = boto3.resource('s3')
        BUCKET_NAME = s3_dir.split('/')[2]
        PREFIX = '/'.join(s3_dir.split('/')[3:]) + '/'
        KEY = PREFIX + 'session.json'
        obj = s3.Object(BUCKET_NAME, KEY)
        data = obj.get()['Body'].read()
        json_string = data.decode('utf-8')
        session_json = json.loads(json_string)
        session_type = session_json['session_type']
        if session_type == 'STAGE_0' or session_type == 'STAGE_1':
            asset_processed = [asset for asset in assets if f"{asset_raw.name}_processed" in asset.name][-1]
            co_assets_dict[mouse_id]['ophys'][asset_raw.name] = {'session_type':session_type,'raw':asset_raw,"processed":asset_processed} 

    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    co_assets_dict[mouse_id]['ophys'] = {ophys_asset_name:co_assets_dict[mouse_id]['ophys'][ophys_asset_name] for ophys_asset_name in ophys_asset_names[:4]}
    for asset_name in co_assets_dict[mouse_id]['ophys']:
        co_assets_dict[mouse_id]['ophys'][asset_name]['glm'] = [asset for asset in assets if f"{asset_name}_glm" in asset.name][-1]
    
    co_assets_dict[mouse_id]['cortical_zstack'] = {}
    try:
        co_assets_dict[mouse_id]['cortical_zstack']['registered'] = [asset for asset in assets if asset.name.startswith(f'ophys-z-stacks_{mouse_id}') and asset.name.endswith('registered')][-1]
    except IndexError:
        print(f"No registered cortical zstack found for mouse {mouse_id}")
        co_assets_dict[mouse_id]['cortical_zstack']['registered'] = None
    
    try:
        co_assets_dict[mouse_id]['cortical_zstack']['segmented'] = [asset for asset in assets if asset.name.startswith(f'ophys-z-stacks_{mouse_id}') and asset.name.endswith('segmented_cpsam')][-1]
    except IndexError:
        print(f"No segmented cortical zstack found for mouse {mouse_id}")
        co_assets_dict[mouse_id]['cortical_zstack']['segmented'] = None
    
    co_assets_dict[mouse_id]['xenium'] = {}
    co_assets_dict[mouse_id]['xenium']['processed'] = [asset for asset in assets if asset.name.startswith("Xenium_") and asset.name.endswith("_processed")][-1]
    co_assets_dict[mouse_id]['xenium']['cell_types'] = [asset for asset in assets if asset.name.startswith("Xenium_") and asset.name.endswith("_mapped")][-1]

    co_assets_dict[mouse_id]['coregistration'] = {}
    co_assets_dict[mouse_id]['coregistration']['ophys_zstack'] = [asset for asset in assets if 'cortical-zstack-to-fov' in asset.tags][-1]
    co_assets_dict[mouse_id]['coregistration']['xenium_zstack'] = [asset for asset in assets if asset.name.startswith("Xenium_ophys") and 'coregistration' in asset.tags][-1]
    

co_assets = list(np.hstack([[a['raw'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a['processed'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a['glm'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a for a in co_assets_dict[mouse_id]['xenium'].values()]+\
[a for a in co_assets_dict[mouse_id]['cortical_zstack'].values()]+\
[a for a in co_assets_dict[mouse_id]['coregistration'].values()] for mouse_id in xenium_mouse_ids]))


# for asset in co_assets:
#     print(asset.id)


No registered cortical zstack found for mouse 797371


In [6]:
mouse_id = 797371
co_assets_dict[mouse_id]['cortical_zstack']['registered'] = client.data_assets.get_data_asset(data_asset_id="c3b48f98-a0e5-476c-8d05-0f941f8b02d1")

In [7]:
co_assets = list(np.hstack([[a for a in co_assets_dict[mouse_id]['xenium'].values()] for mouse_id in xenium_mouse_ids]))
for asset in co_assets:
    print(asset.id)

40328472-ed8e-4f04-98f0-06751cd1899a
b5e8f877-e263-436f-afc6-9e42d933615e
b942d5d8-daa4-411f-aed9-4e6ec082b691
103c25fc-36b2-4653-a80e-5a5033346922
5c76d90b-c4fd-4d0c-9e37-243768060d3d
d9c35976-e2ff-4f3a-bea9-4e5a06b6d238


In [ ]:
data_info_xenium = {}

for mouse_id in xenium_mouse_ids:
    data_info_xenium[mouse_id] = {}
    data_info_xenium[mouse_id]['ophys'] = {}
    data_info_xenium[mouse_id]['xenium'] = {}
    data_info_xenium[mouse_id]['coregistration'] = {}
    data_info_xenium[mouse_id]['cortical_zstack'] = {}


    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    data_info_xenium[mouse_id]['ophys'] = {f"session_{session_ind}":co_assets_dict[mouse_id]['ophys'][ophys_asset_name].copy() for session_ind, ophys_asset_name in enumerate(ophys_asset_names)}
    for session in data_info_xenium[mouse_id]['ophys'].keys():
        data_info_xenium[mouse_id]['ophys'][session]['raw'] = data_info_xenium[mouse_id]['ophys'][session]['raw'].name
        data_info_xenium[mouse_id]['ophys'][session]['processed'] = data_info_xenium[mouse_id]['ophys'][session]['processed'].name
        data_info_xenium[mouse_id]['ophys'][session]['glm'] = data_info_xenium[mouse_id]['ophys'][session]['glm'].name

    data_info_xenium[mouse_id]['xenium']['processed'] = co_assets_dict[mouse_id]['xenium']['processed'].name
    data_info_xenium[mouse_id]['xenium']['cell_types'] = co_assets_dict[mouse_id]['xenium']['cell_types'].name
    data_info_xenium[mouse_id]['coregistration']['ophys_zstack'] = co_assets_dict[mouse_id]['coregistration']['ophys_zstack'].name
    data_info_xenium[mouse_id]['coregistration']['xenium_zstack'] = co_assets_dict[mouse_id]['coregistration']['xenium_zstack'].name
    data_info_xenium[mouse_id]['cortical_zstack']['registered'] = co_assets_dict[mouse_id]['cortical_zstack']['registered'].name
    data_info_xenium[mouse_id]['cortical_zstack']['segmented'] = co_assets_dict[mouse_id]['cortical_zstack']['segmented'].name

data_info_xenium
with open('/root/capsule/code/Preprocessing/data_info_xenium.pkl', 'wb') as f:
    pickle.dump(data_info_xenium, f)
